# Transformer Architecture


In [20]:
%pip install transformers torch

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

## STEP 1: LOAD TOKENIZER AND PRE-TRAINED TRANSFORMER

In [7]:
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

 ## STEP 2: INPUT TEXT

In [8]:
text = "The sun rises in the"


print("=" * 60)
print("STEP 1 - INPUT TEXT")
print("=" * 60)


print(text)


STEP 1 - INPUT TEXT
The sun rises in the


## STEP 3: TOKENIZATION

In [9]:
inputs = tokenizer(text, return_tensors="pt")
token_ids = inputs["input_ids"]
tokens = tokenizer.convert_ids_to_tokens(token_ids[0])
print("\n" + "=" * 60)
print("STEP 2 - TOKENIZATION")
print("=" * 60)
print("Tokens:")
print(tokens)


STEP 2 - TOKENIZATION
Tokens:
['The', 'Ġsun', 'Ġrises', 'Ġin', 'Ġthe']


## Token Id's

In [10]:
print("\n" + "=" * 60)
print("STEP 3 - TOKEN IDs")
print("=" * 60)


print(token_ids)


print("\nToken → Token ID")


for token, token_id in zip(tokens, token_ids[0]):
    print(f"{token:<10} → {token_id.item()}")



STEP 3 - TOKEN IDs
tensor([[  464,  4252, 16736,   287,   262]])

Token → Token ID
The        → 464
Ġsun       → 4252
Ġrises     → 16736
Ġin        → 287
Ġthe       → 262


## Embeddings

In [11]:
with torch.no_grad():
    embeddings = model.transformer.wte(token_ids)


print("\n" + "=" * 60)
print("STEP 4 - EMBEDDINGS")
print("=" * 60)


print("Embedding shape:")
print(embeddings.shape)


print("\nFirst token embedding:")
print(embeddings[0][0])


print("\nNote:")
print("Each token is converted into a numerical vector.")



STEP 4 - EMBEDDINGS
Embedding shape:
torch.Size([1, 5, 768])

First token embedding:
tensor([-6.2649e-02, -4.4906e-02,  5.5888e-02, -5.4657e-02, -1.1713e-01,
        -7.2870e-02, -2.2326e-01, -3.2198e-03,  6.8535e-03,  2.3608e-02,
        -5.8700e-02,  4.4439e-02,  7.4774e-02, -1.3818e-02,  1.1873e-01,
        -5.1842e-02,  5.4415e-02,  5.5382e-02, -3.3126e-02,  1.1923e-01,
        -7.3283e-02,  2.6658e-02, -8.4261e-02,  5.7980e-02, -4.2860e-03,
        -4.0704e-02,  6.5652e-02, -6.6221e-02, -1.0232e-01,  3.1356e-02,
        -1.8873e-02,  2.7774e-02, -2.0423e-02,  1.1994e-01, -8.7572e-02,
        -1.0579e-01, -3.1816e-01,  9.5807e-02,  1.1588e-01, -4.2873e-02,
         1.3187e-01, -1.3457e-01, -1.0421e-01, -1.2150e-01,  9.2551e-02,
        -2.7394e-02,  3.1406e-02,  6.4891e-03,  1.2296e-01, -2.0581e-01,
        -6.3499e-02,  5.4726e-02,  6.0272e-02,  1.1968e-01,  6.4859e-02,
        -3.4885e-01, -5.6065e-02, -2.4721e-03,  3.7549e-03, -1.2817e-02,
        -8.4894e-02, -1.6269e-02,  8.2

## Step 6 Position Information

In [12]:
print("\n" + "=" * 60)
print("STEP 5 - POSITION INFORMATION")
print("=" * 60)


print("Token positions:")


for position, token in enumerate(tokens):
    print(f"Position {position} → {token}")


print("\nThe Transformer also uses position information")
print("to understand the order of tokens.")



STEP 5 - POSITION INFORMATION
Token positions:
Position 0 → The
Position 1 → Ġsun
Position 2 → Ġrises
Position 3 → Ġin
Position 4 → Ġthe

The Transformer also uses position information
to understand the order of tokens.


## STEP 7: TRANSFORMER PROCESSING

In [14]:

with torch.no_grad():
    outputs = model(**inputs)


logits = outputs.logits


print("\n" + "=" * 60)
print("STEP 6 - TRANSFORMER")
print("=" * 60)


print("The input passes through Transformer layers.")


print("\nConceptually:")
print("Embedding + Position")
print("        ↓")
print("Self-Attention")
print("        ↓")
print("Feed-Forward Network")
print("        ↓")
print("Transformer Layers")



STEP 6 - TRANSFORMER
The input passes through Transformer layers.

Conceptually:
Embedding + Position
        ↓
Self-Attention
        ↓
Feed-Forward Network
        ↓
Transformer Layers


## Step8 Logits

In [15]:
print("\n" + "=" * 60)
print("STEP 7 - LOGITS")
print("=" * 60)


print("Logits shape:")
print(logits.shape)


# Get logits for the last input token
next_token_logits = logits[:, -1, :]


print("\nNext-token logits shape:")
print(next_token_logits.shape)


print("\nLogits are raw scores for all possible vocabulary tokens.")



STEP 7 - LOGITS
Logits shape:
torch.Size([1, 5, 50257])

Next-token logits shape:
torch.Size([1, 50257])

Logits are raw scores for all possible vocabulary tokens.


## Step9 SoftMax

In [16]:
probabilities = torch.softmax(next_token_logits, dim=-1)


print("\n" + "=" * 60)
print("STEP 8 - SOFTMAX")
print("=" * 60)


print("Softmax converts logits into probabilities.")



STEP 8 - SOFTMAX
Softmax converts logits into probabilities.


## Step 10 Top 5 probabilities

In [17]:
top_probs, top_token_ids = torch.topk(probabilities, 5)


print("\n" + "=" * 60)
print("STEP 9 - TOP 5 NEXT-TOKEN PROBABILITIES")
print("=" * 60)


for prob, token_id in zip(top_probs[0], top_token_ids[0]):


    token = tokenizer.decode([token_id.item()])


    print(f"{token:<15} → {prob.item() * 100:.2f}%")



STEP 9 - TOP 5 NEXT-TOKEN PROBABILITIES
 sky            → 15.89%
 morning        → 4.67%
 middle         → 4.15%
 air            → 2.95%
 sun            → 2.79%


## Step 11 Select next Token

In [18]:
next_token_id = torch.argmax(probabilities, dim=-1)


print("\n" + "=" * 60)
print("STEP 10 - SELECT NEXT TOKEN")
print("=" * 60)


print("Selected Token ID:")
print(next_token_id.item())



STEP 10 - SELECT NEXT TOKEN
Selected Token ID:
6766


## step 12: Token ID -> Text

In [19]:
next_token = tokenizer.decode(next_token_id)


print("\n" + "=" * 60)
print("STEP 11 - TOKEN ID → TEXT")
print("=" * 60)


print("Predicted next token:")




print("Final Prediction:", next_token)



STEP 11 - TOKEN ID → TEXT
Predicted next token:
Final Prediction:  sky


In [ ]:
hhh